# 03 · Dense retrieval: MiniLM off the shelf (Prompt 6) and the DPR comparison (Prompt 7), owner: M2

Before running:
1. **Accelerator: GPU T4 x1** (preferred over P100; recent PyTorch builds may not support the P100). **Internet: On**.
2. **Add Input**:
   - `fever-corpus-v1` (Your Work → Datasets)
   - the output of notebook **02 · BM25** (Your Work → Notebooks), which supplies `outputs/bm25/run_test.jsonl`
     for the side-by-side examples and the DPR candidate pool.

Expected run time: about 15-30 minutes of GPU (MiniLM over 308k sentences takes a few minutes;
DPR over the candidate pool takes longer, since BERT-base is about 10x larger).

When it finishes: **Save Version → Save & Run All (Commit)**. Prompts 8-9 use this output (embeddings,
index and run files).

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyyaml pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Unit tests

In [ ]:
!python -m pytest -q -rs

## Prompt 6 · MiniLM off the shelf: encode corpus, FAISS Flat index, retrieve, compare with BM25

In [ ]:
!python -m src.retrieval.dense --config configs/dense_offshelf.yaml

In [ ]:
import json, pandas as pd
m = json.load(open("outputs/dense_offshelf/metrics.json"))
print(f"{m['model']} on {m['device']} | index {m['index_size_gb']} GB | {m['ms_per_query']} ms/query | "
      f"corpus encoded in {m['corpus_encode_seconds']} s")
bm = json.load(open(next(__import__("glob").iglob("/kaggle/input/**/outputs/bm25/metrics.json", recursive=True))))
rows = {}
for name, mm in (("BM25", bm), ("MiniLM off-the-shelf", m)):
    for lv in ("page", "sentence"):
        rows[(name, lv)] = {k: v for k, v in mm["test"][lv].items() if not k.startswith("evidence_set")}
pd.DataFrame(rows).T.round(4)

## Prompt 7 · DPR checkpoint vs MiniLM on the 5k test subset

In [ ]:
!python -m src.retrieval.dpr_compare --config configs/dpr_compare.yaml

In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/dpr_compare/comparison.md').read())

In [ ]:
!du -sh outputs/*